In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.types import IntegerType, FloatType

spark.sql("USE CATALOG retail_transactions")

In [0]:
# Definiendo la ruta del archivo
file_path = "/Volumes/retail_transactions/default/retail_raw_data/Retail_Transaction_Dataset.csv"

# Leyendo el csv
df_bronze = (spark.read
    .format("csv")
    .option("header", "true")       
    .option("delimiter", ",")
    .option("quote", '"')           # Maneja las direcciones entre comillas
    .option("escape", '"')          # Maneja comillas dentro de la dirección
    .option("multiLine", "true")    # Vital si hay saltos de línea ocultos en las direcciones
    .option("inferSchema", "false")
    .load(file_path)
)

# Guardando la tabla en fomrmato delta
df_bronze.write.format("delta").mode("overwrite").saveAsTable("retail_transactions.bronze.retail_raw_data")

# Verificando la carga
print(f"Capa bronze completada. Registros cargados: {df_bronze.count()}")
display(df_bronze.limit(5))

In [0]:
# 1. Cargando la tabla Bronze 
df_silver = spark.table("retail_transactions.bronze.retail_raw_data")

# 2. Aplicamos la limpieza y transformaciones
df_silver_clean = df_silver.select(
    F.trim(F.col("CustomerID")).cast(IntegerType()).alias("CustomerID"),
    F.upper(F.trim(F.col("ProductID"))).alias("ProductID"),
    F.trim(F.col("Quantity")).cast(IntegerType()).alias("Quantity"),
    F.col("Price").cast(FloatType()).alias("Price"),
    
    # CAMBIO CLAVE: Usamos M, d y H (un solo dígito) para que acepte '8/15/2023 4:24'
    F.to_timestamp(F.col("TransactionDate"), "M/d/yyyy H:mm").alias("TRANSACTION_TIMESTAMP"),
    
    F.coalesce(F.upper(F.trim(F.col("PaymentMethod"))), F.lit('UNKNOWN')).alias("PAYMENT_METHOD"),
    F.trim(F.col("StoreLocation")).alias("FULL_ADDRESS"),
    
    F.coalesce(
        F.regexp_extract(F.col("StoreLocation"), r' ([A-Z]{2}) [0-9]{5}$', 1),
        F.lit('UNKNOWN')
    ).alias("Store_State"),
    
    F.upper(F.trim(F.col("ProductCategory"))).alias("PRODUCT_CATEGORY"),
    F.col("DiscountApplied").cast(FloatType()).alias("DISCOUNT_PERCENT"),
    F.col("TotalAmount").cast(FloatType()).alias("TOTAL_AMOUNT"),
    F.current_timestamp().alias("LOAD_TIMESTAMP")
).filter(
    F.col("CustomerID").isNotNull() & 
    F.col("TRANSACTION_TIMESTAMP").isNotNull()
)

# 3. Guardando la tabla en formato DELTA (En el esquema SILVER)
# Corregido: Usamos df_silver_clean y la ruta de silver
df_silver_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail_transactions.silver.cleaned_retail_data_py")

print(f"Capa Silver completada. Se procesaron {df_silver_clean.count()} registros limpios.")
display(df_silver_clean.limit(10))

In [0]:
# 1. Cargamos la tabla Silver
df_silver = spark.table("retail_transactions.silver.cleaned_retail_data_py")

# --- INSIGHT 1: Rentabilidad por Categoría ---
# Cambiamos group_by -> groupBy
df_silver.groupBy("PRODUCT_CATEGORY").agg(
    F.round(F.sum("TOTAL_AMOUNT"), 2).alias("TOTAL_REVENUE"),
    F.round(F.avg("DISCOUNT_PERCENT"), 2).alias("AVG_DISCOUNT_PERCENT")
).sort(F.col("TOTAL_REVENUE").desc()
).write.mode("overwrite").saveAsTable("retail_transactions.gold.vw_rentabilidad_por_categoria_py")

# --- INSIGHT 2: Rendimiento Geográfico ---
df_silver.groupBy("Store_State").agg(
    F.round(F.sum("TOTAL_AMOUNT"), 2).alias("TOTAL_REVENUE"),
    F.count("*").alias("TRANSACTIONS_QTY")
).sort(F.col("TOTAL_REVENUE").desc()
).write.mode("overwrite").saveAsTable("retail_transactions.gold.vw_rendimiento_geografico_py")

# --- INSIGHT 3: Tendencia Temporal ---
df_silver.withColumn("TRANSACTION_MONTH", F.date_trunc("month", F.col("TRANSACTION_TIMESTAMP"))) \
    .groupBy("TRANSACTION_MONTH").agg(
    F.round(F.sum("TOTAL_AMOUNT"), 2).alias("TOTAL_REVENUE")
).sort("TRANSACTION_MONTH"
).write.mode("overwrite").saveAsTable("retail_transactions.gold.vw_tendencia_temporal_py")

# --- INSIGHT 4: Análisis de Métodos de Pago ---
df_silver.groupBy("PAYMENT_METHOD").agg(
    F.round(F.sum("TOTAL_AMOUNT"), 2).alias("TOTAL_REVENUE"),
    F.round(F.avg("TOTAL_AMOUNT"), 2).alias("AVG_TICKET_VALUE")
).sort(F.col("TOTAL_REVENUE").desc()
).write.mode("overwrite").saveAsTable("retail_transactions.gold.vw_metodo_pago_analisis_py")

# --- INSIGHT 5: Efectividad del Descuento ---
df_silver.withColumn("DISCOUNT_RANGE", 
    F.when(F.col("DISCOUNT_PERCENT") < 5, "0 - 5%")
     .when(F.col("DISCOUNT_PERCENT") < 10, "5 - 10%")
     .when(F.col("DISCOUNT_PERCENT") < 15, "10 - 15%")
     .otherwise("15 - 20%")
).groupBy("DISCOUNT_RANGE").agg(
    F.sum("Quantity").alias("TOTAL_UNITS_SOLD")
).sort("DISCOUNT_RANGE"
).write.mode("overwrite").saveAsTable("retail_transactions.gold.vw_efectividad_descuento_py")

# --- INSIGHT 6: Ranking de Productos "Best-Sellers" ---
df_silver.groupBy("ProductID").agg(
    F.sum("Quantity").alias("TOTAL_QTY_SOLD"),
    F.round(F.sum("TOTAL_AMOUNT"), 2).alias("TOTAL_REVENUE")
).sort(F.col("TOTAL_REVENUE").desc()
).limit(10) \
.write.mode("overwrite").saveAsTable("retail_transactions.gold.vw_ranking_products_py")

# --- INSIGHT 7: KPI de Canasta Promedio ---
df_silver.groupBy("PRODUCT_CATEGORY").agg(
    F.round(F.avg("Quantity"), 2).alias("AVG_BASKET_SIZE")
).write.mode("overwrite").saveAsTable("retail_transactions.gold.vw_kpi_canasta_promedio_py")

print("Capa Gold terminada. Vistas creadas con éxito.")